# wmh-multisite — test de faisabilité (Kaggle)

Objectif : vérifier, **avant d'écrire le projet**, que les briques lourdes tournent sur un GPU Kaggle et mesurer leur coût.

| Test | Question |
|---|---|
| 1. Environnement | GPU, RAM, CPU, disque disponibles ? |
| 2. Données | Le téléchargement DataverseNL (WMH Challenge 2017) fonctionne depuis Kaggle ? |
| 3. ANTs | Recalage rigide T1→FLAIR et SyN T1→MNI : temps et qualité ? |
| 4. HD-BET | Extraction du cerveau sur GPU : temps, VRAM ? |
| 5. WMH-SynthSeg | Tient-il en VRAM (T4 16 Go) ? sinon sur CPU ? Dice sur 5 sujets ? |
| 6. nnU-Net v2 | Prétraitement des 60 cas + 5 epochs : durée d'une epoch, VRAM, plan choisi ? |
| 7. Résumé | Un JSON à copier-coller dans la conversation |

## Instructions

1. Sur kaggle.com : **Create → New Notebook**, puis **File → Import Notebook** et choisir ce fichier `.ipynb`.
2. Panneau de droite → **Settings** :
   - **Accelerator : GPU T4 x2** (ou T4 x1). *Éviter P100* (architecture ancienne, support PyTorch incertain).
   - **Internet : On** (nécessite un compte vérifié par téléphone).
3. **Run All** (durée totale estimée : 45–75 min). Les tests sont protégés par un *watchdog* (RAM / temps) : une étape qui échoue est notée et le notebook continue.
4. À la fin, **copier la sortie de la dernière cellule** (bloc JSON) dans la conversation. Le fichier `/kaggle/working/feasibility_report.json` est aussi sauvegardé.

> Les données (WMH Challenge, licence CC BY-NC 4.0, DOI 10.34894/AECRSD) sont téléchargées dans `/tmp`, jamais publiées.

In [ ]:
# 1. Environnement
import os, sys, json, time, shutil, subprocess, platform
REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

import torch, psutil
REPORT["env"] = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "gpu_mem_gb": [round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1) for i in range(torch.cuda.device_count())],
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "cpus": os.cpu_count(),
    "disk_tmp_free_gb": round(shutil.disk_usage("/tmp").free / 1e9, 1),
    "disk_working_free_gb": round(shutil.disk_usage("/kaggle/working").free / 1e9, 1),
}
print(json.dumps(REPORT["env"], indent=2))
assert torch.cuda.is_available(), "Pas de GPU : activer 'Accelerator: GPU T4' dans Settings"
x = torch.randn(1024, 1024, device="cuda"); print("Test CUDA OK:", float((x @ x).sum()) is not None)

In [ ]:
# Installation des dépendances (torch est déjà fourni par Kaggle)
!pip install -q antspyx nibabel SimpleITK nnunetv2 hd-bet h5py 2>&1 | tail -3
import numpy as np, nibabel as nib, SimpleITK as sitk, ants
print("numpy", np.__version__, "| antspyx", ants.__version__, "| nibabel", nib.__version__)
REPORT["env"]["versions"] = {"numpy": np.__version__, "antspyx": ants.__version__, "nnunetv2": sh("pip show nnunetv2 | grep Version")}

In [ ]:
# Watchdog : lance une commande, suit RAM (processus + enfants) et VRAM (nvidia-smi), tue si dépassement
import threading
RAM_LIMIT_GB = round(0.85 * psutil.virtual_memory().total / 1e9, 1)

def gpu_used_gb():
    out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits")
    try: return sum(float(v) for v in out.split()) / 1024
    except ValueError: return 0.0

def run_guarded(cmd, timeout_s=1800, ram_limit_gb=RAM_LIMIT_GB, cwd=None, env=None, log="/tmp/guarded.log"):
    t0 = time.time(); gpu0 = gpu_used_gb()
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, cwd=cwd, env=env)
    pp = psutil.Process(p.pid); peak_ram = peak_gpu = 0.0; killed = None
    while p.poll() is None:
        try: rss = sum(c.memory_info().rss for c in [pp] + pp.children(recursive=True)) / 1e9
        except psutil.Error: rss = 0.0
        peak_ram = max(peak_ram, rss); peak_gpu = max(peak_gpu, gpu_used_gb() - gpu0)
        if rss > ram_limit_gb: killed = f"RAM {rss:.1f} GB > {ram_limit_gb} GB"
        elif time.time() - t0 > timeout_s: killed = f"timeout {timeout_s}s"
        if killed:
            for c in pp.children(recursive=True) + [pp]:
                try: c.kill()
                except psutil.Error: pass
            break
        time.sleep(1)
    rc = p.wait()
    tail = open(log).read()[-1500:]
    return {"rc": rc, "seconds": round(time.time() - t0, 1), "peak_ram_gb": round(peak_ram, 1),
            "peak_vram_gb": round(peak_gpu, 1), "killed": killed, "log_tail": tail}
print("Limite RAM du watchdog :", RAM_LIMIT_GB, "GB")

In [ ]:
# 2. Téléchargement des données (API DataverseNL, fichier par fichier)
import urllib.request
DATA = "/tmp/wmh"; os.makedirs(DATA, exist_ok=True)
API = "https://dataverse.nl/api/datasets/:persistentId/?persistentId=doi:10.34894/AECRSD"
FILES = json.load(urllib.request.urlopen(API, timeout=60))["data"]["latestVersion"]["files"]
print(len(FILES), "fichiers dans le dataset")

def download(subject_dirs, keep):
    """subject_dirs: 'training/Utrecht/0'... ; keep(relpath_in_subject) -> bool"""
    n = 0; t0 = time.time()
    for f in FILES:
        d = f.get("directoryLabel") or ""
        for s in subject_dirs:
            if d == s or d.startswith(s + "/"):
                rel = (d[len(s):].strip("/") + "/" + f["dataFile"]["filename"]).strip("/")
                if not keep(rel): continue
                out = os.path.join(DATA, s.replace(" ", "_"), rel)
                if not os.path.exists(out):
                    os.makedirs(os.path.dirname(out), exist_ok=True)
                    urllib.request.urlretrieve(f"https://dataverse.nl/api/access/datafile/{f['dataFile']['id']}", out)
                n += 1
    return n, round(time.time() - t0, 1)

SAMPLE = ["training/Utrecht/0", "training/Singapore/50", "training/Amsterdam/GE3T/100",
          "test/Amsterdam/GE1T5/150", "test/Amsterdam/Philips_VU .PETMR_01./160"]
n, sec = download(SAMPLE, keep=lambda r: True)
REPORT["download_sample"] = {"files": n, "seconds": sec, "size_mb": round(float(sh(f"du -sm {DATA}").split()[0]), 0)}
print(REPORT["download_sample"])

In [ ]:
# 3. ANTs : recalage rigide 3DT1 -> FLAIR (comparé au recalage elastix fourni par le challenge) + SyN T1 -> MNI
def sdir(s): return os.path.join(DATA, s.replace(" ", "_"))
res = []
for s in SAMPLE:
    t0 = time.time()
    fl = ants.image_read(f"{sdir(s)}/orig/FLAIR.nii.gz"); t1 = ants.image_read(f"{sdir(s)}/orig/3DT1.nii.gz")
    reg = ants.registration(fixed=fl, moving=t1, type_of_transform="Rigid")
    ref = ants.image_read(f"{sdir(s)}/orig/T1.nii.gz").numpy(); ours = reg["warpedmovout"].numpy()
    m = ref > np.percentile(ref, 50)
    res.append({"subject": s, "rigid_s": round(time.time() - t0, 1),
                "corr_vs_elastix": round(float(np.corrcoef(ours[m], ref[m])[0, 1]), 3)})
    print(res[-1])
REPORT["ants_rigid"] = res

try:
    t0 = time.time()
    mni = ants.image_read(ants.get_ants_data("mni"))
    t1 = ants.image_read(f"{sdir(SAMPLE[0])}/pre/3DT1.nii.gz")
    syn = ants.registration(fixed=mni, moving=t1, type_of_transform="SyN")
    REPORT["ants_syn_mni"] = {"seconds": round(time.time() - t0, 1), "ok": True}
except Exception as e:
    REPORT["ants_syn_mni"] = {"ok": False, "error": str(e)[:300]}
print(REPORT["ants_syn_mni"])

In [ ]:
# 4. HD-BET (extraction du cerveau, GPU) sur la 3DT1 d'un sujet
src = f"{sdir(SAMPLE[0])}/orig/3DT1.nii.gz"
r = run_guarded(f"hd-bet -i {src} -o /tmp/hdbet_out.nii.gz -device cuda --disable_tta", timeout_s=900)
ok = os.path.exists("/tmp/hdbet_out.nii.gz")
if not ok: print(sh("hd-bet -h")[:1500])  # l'API CLI a changé entre versions : afficher l'aide si échec
REPORT["hd_bet"] = {k: v for k, v in r.items() if k != "log_tail"} | {"ok": ok}
print(REPORT["hd_bet"]); print(r["log_tail"][-600:])

In [ ]:
# 5. WMH-SynthSeg (FreeSurfer, PyTorch) en autonome : code depuis GitHub + modèle (790 Mo) depuis le FTP MGH
WS = "/tmp/wmhss"; os.makedirs(f"{WS}/unet3d", exist_ok=True); os.makedirs("/tmp/fakefs/models", exist_ok=True)
B = "https://raw.githubusercontent.com/freesurfer/freesurfer/dev/mri_WMHsynthseg/WMHSynthSeg"
for f in ["inference.py", "utils.py"]: urllib.request.urlretrieve(f"{B}/{f}", f"{WS}/{f}")
for f in ["__init__.py", "buildingblocks.py", "model.py", "se.py", "utils.py"]: urllib.request.urlretrieve(f"{B}/unet3d/{f}", f"{WS}/unet3d/{f}")
MODEL = "/tmp/fakefs/models/WMH-SynthSeg_v10_231110.pth"
if not os.path.exists(MODEL):
    urllib.request.urlretrieve("https://ftp.nmr.mgh.harvard.edu/pub/dist/lcnpublic/dist/WMH-SynthSeg/WMH-SynthSeg_v10_231110.pth", MODEL)

# Lanceur : autorise les types numpy du checkpoint (PyTorch >= 2.6 charge en weights_only par défaut)
open(f"{WS}/run_wmhss.py", "w").write("""
import runpy, sys, numpy as np, torch
core = getattr(np, "_core", None) or np.core
extra = [getattr(np.dtypes, n) for n in ("Float64DType", "Float32DType", "Int64DType")] if hasattr(np, "dtypes") else []
if hasattr(torch.serialization, "add_safe_globals"):
    try:  # torch >= 2.6: (object, legacy name numpy.core)
        torch.serialization.add_safe_globals([(core.multiarray.scalar, "numpy.core.multiarray.scalar"), np.dtype] + extra)
    except (TypeError, AttributeError):  # torch < 2.6: weights_only=False by default
        pass
sys.argv = ["inference.py"] + sys.argv[1:]
runpy.run_path("inference.py", run_name="__main__")
""")
ENV = dict(os.environ, FREESURFER_HOME="/tmp/fakefs")
print("Code et modèle prêts :", round(os.path.getsize(MODEL) / 1e6), "Mo")

In [ ]:
# 5a. Un sujet : d'abord GPU (--crop), puis repli CPU (--crop) si échec. Le script peut sortir avec rc=0 sans fichier : on vérifie le fichier.
s0 = f"{sdir(SAMPLE[0])}/pre/FLAIR.nii.gz"; os.makedirs("/tmp/wmhss_out", exist_ok=True)
WMHSS = {}
for device, extra in [("cuda", "--crop"), ("cpu", "--crop --threads -1")]:
    out = f"/tmp/wmhss_out/probe_{device}.nii.gz"
    r = run_guarded(f"python run_wmhss.py --i {s0} --o {out} --device {device} {extra}", timeout_s=1200, cwd=WS, env=ENV)
    r["ok"] = os.path.exists(out)
    WMHSS[device] = {k: v for k, v in r.items() if k != "log_tail"}
    print(device, WMHSS[device]); print(r["log_tail"][-500:])
    if r["ok"]: break
REPORT["wmh_synthseg_probe"] = WMHSS

In [ ]:
# 5b. Si un mode fonctionne : les 5 sujets + Dice (label 77 = WMH ; label 2 de la vérité terrain ignoré)
REPORT["wmh_synthseg_dice"] = []
mode = next((d for d, v in WMHSS.items() if v["ok"]), None)
if mode is None:
    print("WMH-SynthSeg ne tourne ni sur GPU ni sur CPU dans cet environnement -> écarté du projet.")
else:
    extra = "--crop" if mode == "cuda" else "--crop --threads -1"
    for s in SAMPLE:
        flair = f"{sdir(s)}/pre/FLAIR.nii.gz"; out = f"/tmp/wmhss_out/{s.replace('/', '_').replace(' ', '_')}.nii.gz"
        r = run_guarded(f"python run_wmhss.py --i {flair} --o {out} --device {mode} {extra}", timeout_s=1200, cwd=WS, env=ENV)
        if not os.path.exists(out):
            REPORT["wmh_synthseg_dice"].append({"subject": s, "error": r["killed"] or r["log_tail"][-200:]}); continue
        pred = ants.resample_image_to_target(ants.image_read(out), ants.image_read(flair), interp_type="nearestNeighbor").numpy() == 77
        gt = nib.load(f"{sdir(s)}/wmh.nii.gz").get_fdata(); ign = gt == 2; g = (gt == 1) & ~ign; p = pred & ~ign
        vox = float(np.prod(nib.load(f"{sdir(s)}/wmh.nii.gz").header.get_zooms())) / 1000
        REPORT["wmh_synthseg_dice"].append({"subject": s, "seconds": r["seconds"], "peak_vram_gb": r["peak_vram_gb"],
            "dice": round(float(2 * (g & p).sum() / max(g.sum() + p.sum(), 1)), 3),
            "vol_gt_ml": round(float(g.sum() * vox), 1), "vol_pred_ml": round(float(p.sum() * vox), 1)})
        print(REPORT["wmh_synthseg_dice"][-1])

In [ ]:
# 6. nnU-Net v2 : conversion des 60 cas d'entraînement (FLAIR + T1 prétraités), plan + prétraitement, 5 epochs
TRAIN = sorted({(f.get("directoryLabel") or "").split("/orig")[0].split("/pre")[0] for f in FILES
                if (f.get("directoryLabel") or "").startswith("training/") and f["dataFile"]["filename"] == "wmh.nii.gz"})
print(len(TRAIN), "sujets d'entraînement")
n, sec = download(TRAIN, keep=lambda r: r in ("pre/FLAIR.nii.gz", "pre/T1.nii.gz", "wmh.nii.gz"))
REPORT["download_train60"] = {"files": n, "seconds": sec}

NN = "/tmp/nnunet"
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.environ[k] = f"{NN}/{k}"; os.makedirs(os.environ[k], exist_ok=True)
os.environ["nnUNet_n_proc_DA"] = str(max(2, (os.cpu_count() or 4) - 1))
D = f"{NN}/nnUNet_raw/Dataset001_WMH"; os.makedirs(f"{D}/imagesTr", exist_ok=True); os.makedirs(f"{D}/labelsTr", exist_ok=True)
for i, s in enumerate(TRAIN):
    cid = f"WMH_{i:03d}"
    shutil.copy(f"{sdir(s)}/pre/FLAIR.nii.gz", f"{D}/imagesTr/{cid}_0000.nii.gz")
    shutil.copy(f"{sdir(s)}/pre/T1.nii.gz", f"{D}/imagesTr/{cid}_0001.nii.gz")
    lab = nib.load(f"{sdir(s)}/wmh.nii.gz")  # labels 0/1/2 en float -> uint8 ; 2 = 'ignore' (doit être le plus grand label)
    nib.save(nib.Nifti1Image(np.rint(lab.get_fdata()).astype(np.uint8), lab.affine), f"{D}/labelsTr/{cid}.nii.gz")
json.dump({"channel_names": {"0": "FLAIR", "1": "T1"}, "labels": {"background": 0, "WMH": 1, "ignore": 2},
           "numTraining": len(TRAIN), "file_ending": ".nii.gz"}, open(f"{D}/dataset.json", "w"), indent=2)

r = run_guarded("nnUNetv2_plan_and_preprocess -d 1 -c 3d_fullres 2d --verify_dataset_integrity", timeout_s=3600)
REPORT["nnunet_preprocess"] = {k: v for k, v in r.items() if k != "log_tail"}
print(REPORT["nnunet_preprocess"]); print(r["log_tail"][-800:])
plans = json.load(open(f"{NN}/nnUNet_preprocessed/Dataset001_WMH/nnUNetPlans.json"))
REPORT["nnunet_plans"] = {c: {k: plans["configurations"][c].get(k) for k in ("patch_size", "batch_size", "spacing")}
                          for c in ("3d_fullres", "2d") if c in plans["configurations"]}
print(json.dumps(REPORT["nnunet_plans"], indent=1))

In [ ]:
# 6b. Entraînement court (5 epochs, fold 0) pour mesurer la durée d'une epoch et la VRAM
import glob, re
REPORT["nnunet_train"] = {}
for cfg in ("3d_fullres", "2d"):
    r = run_guarded(f"nnUNetv2_train 1 {cfg} 0 -tr nnUNetTrainer_5epochs", timeout_s=3600, log=f"/tmp/nn_{cfg}.log")
    logs = glob.glob(f"{NN}/nnUNet_results/Dataset001_WMH/nnUNetTrainer_5epochs__nnUNetPlans__{cfg}/fold_0/training_log_*.txt")
    txt = open(logs[0]).read() if logs else r["log_tail"]
    ep = [float(x) for x in re.findall(r"Epoch time: ([\d.]+) s", txt)]
    dice = re.findall(r"Pseudo dice \[([^\]]+)\]", txt)
    REPORT["nnunet_train"][cfg] = {"rc": r["rc"], "killed": r["killed"], "total_s": r["seconds"], "peak_vram_gb": r["peak_vram_gb"],
        "peak_ram_gb": r["peak_ram_gb"], "epoch_times_s": ep, "last_pseudo_dice": dice[-1] if dice else None,
        "est_hours_250ep": round(np.median(ep) * 250 / 3600, 1) if ep else None,
        "est_hours_1000ep": round(np.median(ep) * 1000 / 3600, 1) if ep else None}
    print(cfg, REPORT["nnunet_train"][cfg])
    if not ep: print(r["log_tail"][-1200:])

In [ ]:
# 7. Résumé : copier tout le bloc JSON ci-dessous dans la conversation
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
json.dump(REPORT, open("/kaggle/working/feasibility_report.json", "w"), indent=1)
print(json.dumps(REPORT, indent=1))